In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic
from rich.pretty import pprint
import json
import anthropic.types as t

load_dotenv()

client = Anthropic()
model = "claude-opus-4-6"

In [10]:
client = Anthropic()

response: t.Message = client.messages.create(
    model="claude-opus-4-7",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Extract the key information from this email: John Smith (john@example.com) is interested in our Enterprise plan and wants to schedule a demo for next Tuesday at 2pm.",
        }
    ],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "email": {"type": "string"},
                    "plan_interest": {"type": "string"},
                    "demo_requested": {"type": "boolean"},
                },
                "required": ["name", "email", "plan_interest", "demo_requested"],
                "additionalProperties": False,
            },
        }
    },
)

valid_json = json.loads(response.content[0].text)
pprint(valid_json)

{'name': 'John Smith', 'email': 'john@example.com', 'plan_interest': 'Enterprise', 'demo_requested': True}

In [20]:
from pydantic import BaseModel, TypeAdapter, Field
from anthropic import Anthropic, transform_schema


class ContactInfo(BaseModel):
    name: str = Field(max_length=100, description="Defines full name of the person")
    email: str
    plan_interest: str
    demo_requested: bool

contact_info_json_schema = transform_schema(TypeAdapter(ContactInfo).json_schema())
pprint(contact_info_json_schema)


{
│   'type': 'object',
│   'title': 'ContactInfo',
│   'properties': {
│   │   'name': {
│   │   │   'type': 'string',
│   │   │   'description': 'Defines full name of the person\n\n{maxLength: 100}',
│   │   │   'title': 'Name'
│   │   },
│   │   'email': {'type': 'string', 'title': 'Email'},
│   │   'plan_interest': {'type': 'string', 'title': 'Plan Interest'},
│   │   'demo_requested': {'type': 'boolean', 'title': 'Demo Requested'}
│   },
│   'additionalProperties': False,
│   'required': ['name', 'email', 'plan_interest', 'demo_requested']
}

In [ ]:
from anthropic import transform_schema
from pydantic import TypeAdapter

# First convert Pydantic model to JSON schema, then transform
schema = TypeAdapter(ContactInfo).json_schema()
schema = transform_schema(schema)
pprint(schema)

response = client.messages.parse(
    model="claude-opus-4-7",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Extract the key information from this email: person interested in our Enterprise plan and wants to schedule a demo for next Tuesday at 2pm.",
        }
    ],
    output_format=ContactInfo
)

pprint(response.parsed_output)

{
│   'type': 'object',
│   'title': 'ContactInfo',
│   'properties': {
│   │   'name': {
│   │   │   'type': 'string',
│   │   │   'description': 'Defines full name of the person\n\n{maxLength: 100}',
│   │   │   'title': 'Name'
│   │   },
│   │   'email': {'type': 'string', 'title': 'Email'},
│   │   'plan_interest': {'type': 'string', 'title': 'Plan Interest'},
│   │   'demo_requested': {'type': 'boolean', 'title': 'Demo Requested'}
│   },
│   'additionalProperties': False,
│   'required': ['name', 'email', 'plan_interest', 'demo_requested']
}

ContactInfo(name='killer', email='', plan_interest='Enterprise', demo_requested=True)

In [16]:
from pydantic import BaseModel

client = Anthropic()


class Classification(BaseModel):
    category: str
    confidence: float
    tags: list[str] | None
    sentiment: str


feedback_text = "Great product, but the delivery was slow."
response = client.messages.parse(
    model="claude-opus-4-7",
    max_tokens=1024,
    output_format=Classification,
    messages=[{"role": "user", "content": f"Classify this feedback: {feedback_text}"}],
)

pprint(response.parsed_output)

Classification(
│   category='Product Feedback',
│   confidence=0.92,
│   tags=['product', 'delivery', 'shipping'],
│   sentiment='mixed'
)

In [19]:
response = client.messages.create(
    model="claude-opus-4-7",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Help me plan a trip to Paris departing May 15, 2026",
        }
    ],
    # JSON outputs: structured response format
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string"},
                    "next_steps": {"type": "array", "items": {"type": "string"}},
                },
                "required": ["summary", "next_steps"],
                "additionalProperties": False,
            },
        }
    },
    # Strict tool use: guaranteed tool parameters
    tools=[
        {
            "name": "search_flights",
            # "strict": True,
            "input_schema": {
                "type": "object",
                "properties": {
                    "destination": {"type": "string"},
                    "date": {"type": "string", "format": "date"},
                },
                "required": ["destination", "date"],
                "additionalProperties": False,
            },
        }
    ],
)

pprint(response)

Message(
│   id='msg_01KqSugmPJzvpqht6ros7uLV',
│   container=None,
│   content=[
│   │   ToolUseBlock(
│   │   │   id='toolu_01FUYgys2nqRS6z84xc1UVNS',
│   │   │   caller=DirectCaller(type='direct'),
│   │   │   input={'destination': 'Paris', 'date': '2026-05-15'},
│   │   │   name='search_flights',
│   │   │   type='tool_use'
│   │   )
│   ],
│   model='claude-opus-4-7',
│   role='assistant',
│   stop_details=None,
│   stop_reason='tool_use',
│   stop_sequence=None,
│   type='message',
│   usage=Usage(
│   │   cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=1028,
│   │   output_tokens=93,
│   │   server_tool_use=None,
│   │   service_tier='standard'
│   )
)